# LLM Evaluation Framework — Phase 1

**Goal of this notebook:** get ONE answer from an LLM and grade its **faithfulness** (is the answer actually supported by the source document, or made up?).

This is the seed of a bigger project. Once this works end-to-end, you'll add more metrics, save results to a database, and build a dashboard. But first: one question, one answer, one score.

**How to use this notebook:** run each cell top to bottom with `Shift + Enter`. Read the notes above each cell so you understand *why*, not just *what*.

> Uses the **Anthropic API** (Claude). If you prefer OpenAI, the same structure works — only the API call changes.

## Step 1 — Install the library

Colab starts fresh every time, so we install what we need. The `-q` flag just keeps the output quiet.

In [ ]:
!pip install anthropic -q
print("Installed.")

## Step 2 — Add your API key (safely)

**Never paste your API key directly into a notebook cell** — if you share the notebook, you leak your key.

Instead, use Colab's built-in **Secrets** manager:
1. Click the **🔑 key icon** in the left sidebar.
2. Click **"Add new secret"**.
3. Name it exactly `ANTHROPIC_API_KEY` and paste your key as the value.
4. Toggle **"Notebook access"** ON.

Then run the cell below. If you don't have a key yet, get one at console.anthropic.com.

In [ ]:
from google.colab import userdata
import anthropic

# Pulls the key from Colab Secrets (not stored in the notebook itself)
API_KEY = userdata.get('ANTHROPIC_API_KEY')

client = anthropic.Anthropic(api_key=API_KEY)
print("Client ready.")

## Step 3 — Configuration

We use two models:
- **TARGET_MODEL** — the model we are *testing* (the "system under test"). This is the one whose answers we grade.
- **JUDGE_MODEL** — a separate model that acts as the *grader*. Using an LLM to grade another LLM's output is called **"LLM-as-judge"** and it's the standard industry technique.

They can be the same model, but keeping them as separate variables means you can swap either one later.

In [ ]:
TARGET_MODEL = "claude-sonnet-4-6"   # the model being tested
JUDGE_MODEL  = "claude-sonnet-4-6"   # the model doing the grading

print("Target model:", TARGET_MODEL)
print("Judge model :", JUDGE_MODEL)

## Step 4 — Load your test dataset (Option B: your own data)

This is *your* domain dataset — the thing that makes the project yours instead of a tutorial clone. Each item has:
- `question` — what we ask the model
- `context` — the source document the answer SHOULD come from
- `reference_answer` — the known-correct answer (we'll use this in later phases)

Below we include a small starter set of 5 medical Q&A items so you can run immediately. **Replace these with your own** (from real PDFs in your chosen domain) as you expand — aim for 30–50 eventually.

Two ways to load your data:
- **Quick:** just edit the list in the cell below.
- **Better:** upload an `eval_dataset.json` file to Colab and load it (code provided, commented out).

In [ ]:
# --- Option 1: dataset defined inline (edit freely) ---
eval_dataset = [
    {
        "id": "q001",
        "question": "What is the recommended adult dosage of amoxicillin for a mild ear infection?",
        "context": "Amoxicillin is a penicillin antibiotic used to treat bacterial infections. For a mild to moderate ear infection in adults, the usual recommended dosage is 500 mg taken orally every 12 hours for 7 days. The medication should be taken with a full glass of water and may be taken with or without food. Patients allergic to penicillin should not take amoxicillin.",
        "reference_answer": "500 mg every 12 hours for 7 days."
    },
    {
        "id": "q002",
        "question": "Who should not take amoxicillin?",
        "context": "Amoxicillin is a penicillin antibiotic used to treat bacterial infections. For a mild to moderate ear infection in adults, the usual recommended dosage is 500 mg taken orally every 12 hours for 7 days. The medication should be taken with a full glass of water and may be taken with or without food. Patients allergic to penicillin should not take amoxicillin.",
        "reference_answer": "Patients who are allergic to penicillin should not take amoxicillin."
    },
    {
        "id": "q003",
        "question": "What lifestyle changes are recommended to lower high blood pressure?",
        "context": "Hypertension, or high blood pressure, can often be managed with lifestyle changes before medication is required. Recommended changes include reducing dietary sodium to less than 1,500 mg per day, engaging in at least 150 minutes of moderate aerobic exercise per week, maintaining a healthy body weight, limiting alcohol intake, and quitting smoking. If blood pressure remains above 140/90 after three months of lifestyle changes, medication may be prescribed.",
        "reference_answer": "Reduce sodium below 1,500 mg per day, exercise at least 150 minutes per week, maintain a healthy weight, limit alcohol, and quit smoking."
    },
    {
        "id": "q004",
        "question": "When might medication be prescribed for high blood pressure?",
        "context": "Hypertension, or high blood pressure, can often be managed with lifestyle changes before medication is required. Recommended changes include reducing dietary sodium to less than 1,500 mg per day, engaging in at least 150 minutes of moderate aerobic exercise per week, maintaining a healthy body weight, limiting alcohol intake, and quitting smoking. If blood pressure remains above 140/90 after three months of lifestyle changes, medication may be prescribed.",
        "reference_answer": "If blood pressure stays above 140/90 after three months of lifestyle changes."
    },
    {
        "id": "q005",
        "question": "What are common early symptoms of type 2 diabetes?",
        "context": "Type 2 diabetes often develops gradually, and early symptoms can be easy to miss. Common early signs include increased thirst, frequent urination, unexplained fatigue, blurred vision, and slow-healing wounds. Some patients also notice increased hunger and tingling in the hands or feet. Early diagnosis through a simple blood glucose test allows for better management and can prevent complications.",
        "reference_answer": "Increased thirst, frequent urination, fatigue, blurred vision, slow-healing wounds, increased hunger, and tingling in the hands or feet."
    }
]

# --- Option 2: load from an uploaded file instead ---
# from google.colab import files
# uploaded = files.upload()          # pick your eval_dataset.json
# import json
# with open('eval_dataset.json') as f:
#     eval_dataset = json.load(f)

print(f"Loaded {len(eval_dataset)} test items.")
print("First question:", eval_dataset[0]['question'])

## Step 5 — Get an answer from the target model

This function is the "system under test." Given a question and a context document, it asks the target model to answer **using only the context**. That instruction matters — it's exactly the RAG setup a real chatbot uses, where the model should stick to retrieved documents and not invent things.

We also capture two things we'll need later:
- **token usage** → for cost (Phase 2)
- **elapsed time** → for latency (Phase 2)

Returning them now means we don't have to rewrite this function later.

In [ ]:
import time

def get_answer(question, context):
    """Ask the target model to answer a question using only the given context.
    Returns the answer text plus token counts and how long the call took."""
    prompt = f"""Answer the question using ONLY the information in the context below.
If the context does not contain the answer, say "I don't know."

Context:
{context}

Question: {question}

Answer:"""

    start = time.time()
    response = client.messages.create(
        model=TARGET_MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    elapsed = time.time() - start

    answer = response.content[0].text.strip()
    usage = {
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }
    return answer, usage, elapsed

## Step 6 — The faithfulness judge (LLM-as-judge)

Now the heart of Phase 1. We ask a **second** model to score how faithful the answer is to the context — meaning, is every claim in the answer actually backed by the source document?

- Score near **1.0** → fully grounded, no made-up information.
- Score near **0.0** → hallucinated, contradicts, or adds facts not in the context.

**Key trick:** we tell the judge to *return only a number*. That makes the response trivial to parse. If you let a judge explain itself, you then have to fish the number out of a paragraph — avoid that early on.

In [ ]:
def judge_faithfulness(answer, context):
    """Use the judge model to score how well the answer is supported by the context.
    Returns a float between 0 and 1."""
    judge_prompt = f"""You are grading whether an answer is faithful to a source context.
Faithful means every claim in the answer is supported by the context, with nothing invented or contradicted.

Context:
{context}

Answer:
{answer}

Score the answer's faithfulness from 0.0 (completely unsupported/hallucinated) to 1.0 (fully supported).
Return ONLY the number, nothing else."""

    response = client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": judge_prompt}],
    )
    raw = response.content[0].text.strip()

    # Parse the number defensively — judges occasionally add stray text.
    try:
        score = float(raw)
    except ValueError:
        import re
        match = re.search(r"[0-1](?:\.\d+)?", raw)
        score = float(match.group()) if match else None
    return score

## Step 7 — Run it on ONE question 🎉

This is your **first milestone**. We take the first item from the dataset, get an answer, grade it, and print everything.

Read the output carefully: does the answer look right? Does the faithfulness score make sense? You now have a working evaluation loop for a single item.

In [ ]:
item = eval_dataset[0]

print("QUESTION:")
print(" ", item["question"])
print()

answer, usage, elapsed = get_answer(item["question"], item["context"])
print("MODEL'S ANSWER:")
print(" ", answer)
print()

score = judge_faithfulness(answer, item["context"])
print("FAITHFULNESS SCORE:", score)
print()
print(f"(tokens in/out: {usage['input_tokens']}/{usage['output_tokens']}  |  time: {elapsed:.2f}s)")

## Step 8 — Sanity-check the judge (prove it isn't just rubber-stamping)

A grader that gives everything a high score is useless. Let's confirm the judge actually *catches* a bad answer by feeding it a deliberately hallucinated one. You should see a **low** score here — that's the judge working.

In [ ]:
bad_answer = "The recommended dosage is 2000 mg every hour, and it cures all diseases instantly."
bad_score = judge_faithfulness(bad_answer, eval_dataset[0]["context"])

print("Deliberately wrong answer:")
print(" ", bad_answer)
print()
print("Faithfulness score (should be LOW):", bad_score)

## Step 9 — Run across the whole dataset

Same loop, now over every item. This gives you your first *batch* of results and an average faithfulness across the set — a taste of Phase 2, where we'll add more metrics and save everything to a database.

In [ ]:
results = []

for item in eval_dataset:
    answer, usage, elapsed = get_answer(item["question"], item["context"])
    score = judge_faithfulness(answer, item["context"])
    results.append({
        "id": item["id"],
        "question": item["question"],
        "answer": answer,
        "faithfulness": score,
    })
    print(f"{item['id']}: faithfulness = {score}")

scores = [r["faithfulness"] for r in results if r["faithfulness"] is not None]
avg = sum(scores) / len(scores) if scores else 0
print()
print(f"Average faithfulness across {len(scores)} items: {avg:.3f}")

## ✅ Phase 1 complete — what you just built

You now have a working evaluation loop that:
1. Asks a target LLM to answer from a source document.
2. Uses a second LLM as a judge to score faithfulness.
3. Confirms the judge catches bad answers.
4. Runs across a whole dataset and reports an average.

**What's next (Phase 2):**
- Add more judges: **answer relevance** and **correctness** (vs. your `reference_answer`).
- Compute **cost** (from the token counts we already capture) and **latency**.
- Save every run to a **DuckDB** database with a timestamp, so you can track quality **over time**.

**Before moving on:** replace the 5 sample medical items with 30–50 of your own, from real documents in your chosen domain. That's what turns this from a tutorial into *your* project.

**Tip:** Download this notebook to keep it (File → Download → `.ipynb`), and start a GitHub repo now so your work is version-controlled from day one.